# Notebook 02: Kích hoạt SageMaker Training Job với Script Mode (Tuần 2)
**Mô hình:** MobileNetV2 (Transfer Learning PyTorch)  
**Compute Instance:** `ml.c5.xlarge` (CPU, 4 vCPU, 8 GiB RAM)  
**IAM Role:** `arn:aws:iam::<Account_ID>:role/LabRole`  
**Ràng buộc:** Máy train chạy độc lập và tự động giải phóng ngay sau khi kết thúc.

In [ ]:
import boto3
import sagemaker
from sagemaker.pytorch import PyTorch

# 1. Lấy thông tin cấu hình môi trường
account_id = boto3.client('sts').get_caller_identity()['Account']
role = f"arn:aws:iam::{account_id}:role/LabRole"
bucket_name = f"sagemaker-plant-leaf-{account_id}"

s3_train = f"s3://{bucket_name}/dataset/train"
s3_val = f"s3://{bucket_name}/dataset/val"
s3_output = f"s3://{bucket_name}/output"

print(f"IAM Role : {role}")
print(f"S3 Train : {s3_train}")
print(f"S3 Output: {s3_output}")

In [ ]:
# 2. Khởi tạo PyTorch Estimator theo chuẩn SageMaker Script Mode
import os

source_candidates = ['Cloud/src', '../src', 'src']
source_dir = next((d for d in source_candidates if os.path.exists(d)), 'Cloud/src')
print(f'Thư mục mã nguồn huấn luyện: {source_dir}')

estimator = PyTorch(
    entry_point='train.py',
    source_dir=source_dir,
    role=role,
    framework_version='2.1.0',
    py_version='py310',
    instance_count=1,
    instance_type='ml.c5.xlarge',  # CPU instance theo yêu cầu đề bài
    output_path=s3_output,
    hyperparameters={
        'epochs': 8,
        'batch_size': 32,
        'learning_rate': 0.001,
        'model_name': 'mobilenet_v2'
    },
    tags=[
        {'Key': 'Project', 'Value': 'PlantLeafClassification'},
        {'Key': 'Owner', 'Value': 'LeBaTam'}
    ]
)

print('[THÀNH CÔNG] PyTorch Estimator đã được khởi tạo sẵn sàng.')


In [ ]:
# 3. Kích hoạt Training Job trên AWS SageMaker
# Quá trình huấn luyện trên máy ml.c5.xlarge mất khoảng 6-10 phút cho 8 epochs
print("Đang submit Training Job lên AWS SageMaker...")
estimator.fit({'train': s3_train, 'val': s3_val})

In [ ]:
# 4. Xác nhận Model Artifact đã lưu thành công trên S3
print("=" * 60)
print("HUẤN LUYỆN HOÀN TẤT THÀNH CÔNG!")
print(f"Đường dẫn Model Artifact: {estimator.model_data}")
print("=" * 60)